# Neural Network Cloud — Predicción Local + Kaggle Submission

**Corre localmente** después de descargar el modelo entrenado en Kaggle Notebooks.

**Archivos necesarios en la misma carpeta:**
- `nn_cloud_best.keras` — modelo TensorFlow/Keras
- `nn_cloud_preproc.pkl` — preprocesador (imputer + scaler)
- `nn_cloud_metadata.json` — metadata

In [1]:
import tensorflow as tf
import joblib
import pandas as pd
import numpy as np
import json
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

print(f'TensorFlow version: {tf.__version__}')
print('Imports OK')

TensorFlow version: 2.19.0
Imports OK


In [2]:
MODEL_DIR = Path('.')
DATA_DIR  = Path('../../data/processed')

model_file  = MODEL_DIR / 'nn_cloud_best.keras'
preproc_file = MODEL_DIR / 'nn_cloud_preproc.pkl'
meta_file   = MODEL_DIR / 'nn_cloud_metadata.json'

print(f'MODEL_DIR : {MODEL_DIR.resolve()}')
for f in [model_file, preproc_file, meta_file]:
    status = '✓ encontrado' if f.exists() else '✗ NO encontrado — descargar desde Kaggle'
    print(f'  {f.name} : {status}')

MODEL_DIR : C:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\cloud\nn
  nn_cloud_best.keras : ✓ encontrado
  nn_cloud_preproc.pkl : ✓ encontrado
  nn_cloud_metadata.json : ✓ encontrado


In [3]:
with open(meta_file, 'r') as f:
    meta = json.load(f)

print('Metadata del modelo:')
_val_auc = meta.get('val_auc', meta.get('best_cv_auc', 'N/A'))
print(f'  Val AUC (20%)    : {_val_auc}')
print(f'  Train AUC full   : {meta["train_auc"]}')
print(f'  best_epoch       : {meta["best_epoch"]}')
print(f'  n_features       : {len(meta["feature_cols"])}')
print(f'  GPU usado        : {meta["use_gpu"]}')
print(f'  TF version       : {meta["tf_version"]}')
print(f'  Timestamp        : {meta["timestamp"]}')
print('  Hiperparámetros:')
for k, v in meta['best_params'].items():
    print(f'    {k:<16}: {v}')

Metadata del modelo:
  Val AUC (20%)    : 0.767319
  Train AUC full   : 0.777267
  best_epoch       : 33
  n_features       : 30
  GPU usado        : True
  TF version       : 2.19.0
  Timestamp        : 2026-02-25T17:46:29.592040
  Hiperparámetros:
    n_layers        : 3
    units           : 128
    dropout_rate    : 0.4461153150612369
    learning_rate   : 0.004147467608476828
    batch_size      : 1024


In [4]:
print(f'Cargando modelo: {model_file.name}  ({model_file.stat().st_size/1e6:.2f} MB)...')
model = tf.keras.models.load_model(str(model_file))
print('Modelo cargado ✓')
print(f'  Parámetros: {model.count_params():,}')

print(f'\nCargando preprocesador: {preproc_file.name}...')
preproc = joblib.load(preproc_file)
imputer = preproc['imputer']
scaler  = preproc['scaler']
print('Preprocesador cargado ✓')

Cargando modelo: nn_cloud_best.keras  (0.52 MB)...
Modelo cargado ✓
  Parámetros: 38,657

Cargando preprocesador: nn_cloud_preproc.pkl...
Preprocesador cargado ✓


In [5]:
print('Cargando features_test.parquet...')
df_test      = pd.read_parquet(DATA_DIR / 'features_test.parquet')
sk_ids_test  = df_test['SK_ID_CURR'].values
feature_cols = meta['feature_cols']

# Encodear categóricas si existen
cat_cols = [c for c in feature_cols if df_test[c].dtype == 'object']
if cat_cols:
    for col in cat_cols:
        cats    = df_test[col].dropna().unique()
        mapping = {v: i for i, v in enumerate(cats)}
        df_test[col] = df_test[col].map(mapping)

X_test_raw = df_test[feature_cols].values

# Aplicar preprocesamiento
X_test_imp    = imputer.transform(X_test_raw)
X_test_scaled = scaler.transform(X_test_imp)

print(f'  Test shape  : {X_test_scaled.shape}')
print(f'  Features    : {len(feature_cols)}')
print(f'  NaNs en X   : {np.isnan(X_test_scaled).sum():,}')

Cargando features_test.parquet...
  Test shape  : (48744, 30)
  Features    : 30
  NaNs en X   : 0


In [6]:
print('Generando predicciones...')
y_test_prob = model.predict(X_test_scaled, verbose=0).ravel()

print(f'  Predicciones generadas : {len(y_test_prob):,}')
print(f'  Score mínimo           : {y_test_prob.min():.5f}')
print(f'  Score máximo           : {y_test_prob.max():.5f}')
print(f'  Score medio            : {y_test_prob.mean():.5f}')
print(f'  % predicho como default: {(y_test_prob > 0.5).mean()*100:.2f}%')

Generando predicciones...
  Predicciones generadas : 48,744
  Score mínimo           : 0.00096
  Score máximo           : 0.99526
  Score medio            : 0.39776
  % predicho como default: 34.51%


In [7]:
submission = pd.DataFrame({'SK_ID_CURR': sk_ids_test, 'TARGET': y_test_prob})
out_path   = DATA_DIR / 'submission_cloud_nn.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardado: {out_path}')
print(f'Shape: {submission.shape}')
display(submission.head())

Submission guardado: ..\..\data\processed\submission_cloud_nn.csv
Shape: (48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.367839
1,100005,0.788282
2,100013,0.255160
3,100028,0.255580
4,100038,0.658202


## Kaggle Submission — AUC Test (Public Leaderboard)

> **Límite**: 5 submissions/día.

In [12]:
from kaggle import KaggleApi

COMPETITION = 'home-credit-default-risk'
_val_auc = meta.get('val_auc', meta.get('best_cv_auc', 0))
_msg = (f"NN Cloud | "
        f"Val AUC={_val_auc:.5f} | "
        f"train AUC={meta['train_auc']:.5f} | "
        f"layers={meta['best_params']['n_layers']} units={meta['best_params']['units']}")

def _as_str(x):
    return '' if x is None else str(x)

def _get_score(api, comp, file_name, message, poll=10, timeout=300):
    start = time.time()
    while time.time() - start < timeout:
        subs = api.competition_submissions(comp)
        matched = next(
            (s for s in subs
             if _as_str(getattr(s, 'file_name', None)) == file_name
             and _as_str(getattr(s, 'description', None)) == message),
            subs[0] if subs else None
        )
        if matched:
            pub    = _as_str(getattr(matched, 'public_score',  None))
            status = _as_str(getattr(matched, 'status',        None))
            elapsed = int(time.time() - start)
            print(f'  [{elapsed:>3}s] status={status!r}  public_score={pub!r}')
            if pub and pub.lower() not in ('', 'none', 'null', '-'):
                priv = _as_str(getattr(matched, 'private_score', None))
                return float(pub), (float(priv) if priv and priv.lower() not in ('','none','null','-') else None)
        time.sleep(poll)
    return None, None

_api = KaggleApi()
_api.authenticate()

print(f'Enviando: {_msg}')
_api.competition_submit(file_name=str(out_path), message=_msg, competition=COMPETITION)
print('Esperando scoring (poll 10 s / máx 5 min)...')

public_auc, private_auc = _get_score(_api, COMPETITION, out_path.name, _msg)

print(f'\n' + '=' * 65)
print(f'RESULTADO KAGGLE — NEURAL NETWORK CLOUD')
print(f'=' * 65)
print(f'  AUC test Public  LB  : {public_auc}')
print(f'  AUC test Private LB  : {private_auc}')
print(f'  Val AUC (entrenamiento): {_val_auc:.5f}')
if public_auc:
    print(f'  Gap Val - Public LB  : {abs(_val_auc - public_auc):.5f}')
print(f'=' * 65)

Enviando: NN Cloud | Val AUC=0.76732 | train AUC=0.77727 | layers=3 units=128


100%|██████████| 888k/888k [00:01<00:00, 579kB/s]  


Esperando scoring (poll 10 s / máx 5 min)...
  [  0s] status='SubmissionStatus.PENDING'  public_score=''
  [ 10s] status='SubmissionStatus.COMPLETE'  public_score='0.76479'

RESULTADO KAGGLE — NEURAL NETWORK CLOUD
  AUC test Public  LB  : 0.76479
  AUC test Private LB  : 0.75308
  Val AUC (entrenamiento): 0.76732
  Gap Val - Public LB  : 0.00253
